# Diseño y benchmarking de la arquitectura de ingesta de datos OHLCV

La adquisición de datos financieros históricos de alta calidad es el cimiento de cualquier investigación cuantitativa. En este cuaderno realizamos un *benchmarking* técnico de las distintas metodologías de conexión con la API de Binance (`requests`, `python-binance` y `ccxt`). El análisis no se limita a la descarga: evaluamos latencia, gestión de límites de tasa (*rate limits*) y resiliencia ante fallos de cada enfoque. El objetivo es justificar la arquitectura elegida para el ETL productivo.

**Relación con el resto del pilar OHLCV.** La evidencia formal de ingesta (validador, SQL, exportación versionada) vive en `02_evidencia_ingesta_ohlcv.ipynb` (`notebooks/01_ingesta/ohlcv/`). Este cuaderno es **exploratorio y pedagógico**: no escribe bajo `reports/`; los `.csv` de prueba se generan en el directorio de trabajo del kernel (`{SYMBOL}_4h_{origen}.csv`).

## Binance API (REST)

En este primer enfoque, interactuaremos con la API al nivel más bajo posible (HTTP puro). No utilizaremos librerías intermediarias específicas de criptomonedas, sino herramientas estándar de Python para peticiones web (`requests`).

### Documentación y endpoint


Trabajaremos con la API spot (v3) de Binance.
- **Documentación oficial:** https://developers.binance.com/docs/binance-spot-api-docs/rest-api
- **Endpoint base:** `https://api.binance.com/api/v3/klines`


### Importación y parámetros:

**Importación de librerías:**
Comenzamos importando las librerías que necesitamos para el funcionamiento del script:
- `requests`: Para realizar las peticiones HTTP a la API de Binance.
- `pandas`: Para estructurar datos recibidos en un DataFrame y exportarlos cómodamente a un archivo `.csv`.
- `time`: Nos permite gestionar las pausas (`sleep`) para respetar los límites de la API y medir los rendimientos de los scripts.
- `datetime`: Fundamental para convertir entre milisegundos (valor en el que opera la API de Binance) y fechas legibles.
- `re`: Necesaria para extraer el tiempo de espera en caso de un bloqueo temporal de IP.
- `IPython.display`: Librería específica de Jupyter para renderizar DataFrames con un formato visual limpio y organizado.

**Parámetros**:
-  `SYMBOL`: Lista con los pares de trading (ej. `BTCUSDT`).
-  `INTERVAL`: Temporalidad de la vela (ej. `4h`).
-  `START_TIME`: Definimos deliberadamente una fecha de inicio (01/01/2017) anterior a la existencia de la mayoría de pares. Esto garantiza que el script capture todo el historial disponible desde el momento exacto en que cada activo comenzó a cotizar en Binance, sin necesidad de conocer esa fecha de antemano.
-  `LIMIT`: Velas por petición. El valor por defecto es 500, pero usaremos el máximo de 1,000 para optimizar la descarga.

In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone
import re
from IPython.display import display

BASE_URL = "https://api.binance.com/api/v3/klines"

SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT", "BNBUSDT"]
INTERVAL = "4h"
LIMIT = 1000

start_date = datetime(2017, 1, 1, tzinfo=timezone.utc)
START_TIME = int(start_date.timestamp() * 1000)

# Reutiliza sesión HTTP en el bucle de descarga
session = requests.Session()

print("Configuración completada. Sesión HTTP iniciada.")

Configuración completada. Sesión HTTP iniciada.


### Comprobación de conectividad y latencia



Antes de iniciar el bucle de descarga, verificamos el servicio utilizando el endpoint de prueba `/api/v3/ping`. Aprovecharemos esta llamada para medir la latencia usando el atributo `.elapsed` de la librería `requests`. Esto nos dará una idea de la velocidad de nuestra conexión con los servidores de Binance.
- **Documentación:**  https://developers.binance.com/docs/binance-spot-api-docs/rest-api/general-endpoints


In [2]:
PING_URL = "https://api.binance.com/api/v3/ping"

print(f"Iniciando prueba de conexión con {PING_URL}.")

try:
    response = session.get(PING_URL)

    response.raise_for_status()

    # Convierte elapsed a milisegundos
    latency_ms = response.elapsed.total_seconds() * 1000

    print("Conexión exitosa.")
    print(f"\t- Latencia: {latency_ms:.2f} ms.")

except requests.exceptions.RequestException as e:
    print(f"Error de conexión: {e}")

Iniciando prueba de conexión con https://api.binance.com/api/v3/ping.
Conexión exitosa.
	- Latencia: 298.50 ms.


### Gestión dinámica de límites de la API 


Para maximizar la eficiencia y evitar bloqueos, nuestro script leerá los límites directamente de la API (`exchangeInfo`) y monitoreará el consumo en tiempo real a través de las cabeceras de respuesta.
- **Endpoint:** `GET /api/v3/exchangeInfo`
- **Documentación:**
    - [Endpoints generales](https://developers.binance.com/docs/binance-spot-api-docs/rest-api/general-endpoints)
    - [Reglas de rate limits](https://developers.binance.com/docs/binance-spot-api-docs/rest-api/limits)
    - [Guía de buenas prácticas: cómo evitar baneos](https://www.binance.com/en/academy/articles/how-to-avoid-getting-banned-by-rate-limits)

Tras consultar `exchangeInfo`, obtendremos los límites vigentes. Nos centraremos en el único que nos afecta:

1. `REQUEST_WEIGHT`: Binance utiliza un sistema de pesos en lugar de un simple contador de peticiones.

    - **Coste por petición:** Solicitar velas (`/api/v3/klines`) en el mercado spot tiene un coste de 1 de peso por petición (independientemente del parámetro `limit` o el número de velas).

    - **Límite estándar:** Habitualmente se asignan 6,000 de peso por minuto por dirección IP.

    - **Mecanismo de control dinámico:** Según la documentación, cada respuesta de la API incluye una cabecera específica que indica el consumo actual: `X-MBX-USED-WEIGHT-(intervalNum)(intervalLetter)`.
        - Nuestro código construirá este nombre dinámicamente (ej. `X-MBX-USED-WEIGHT-1M` para un intervalo de 1 minuto) y pausará la ejecución si el valor devuelto se acerca al límite máximo (ej. > 90% del cupo).
 
2. Gestión de avisos, baneos (429 y 418) y errores de servidor (5XX): Es crítico distinguir entre una advertencia y un bloqueo.

    - **Error 429:** Indica que hemos excedido el límite de peso. La API enviará una cabecera `Retry-After`. Es obligatorio detenerse y esperar los segundos indicados para evitar una escalada.

    - **Error 418:** Ocurre si ignoramos repetidamente los errores 429. Es un bloqueo de IP que escala desde los 2 minutos hasta los 3 días. El script entonces debe detenerse inmediatamente.

        - Obtenemos una respuesta con el siguiente formato:
          ```json
          {
              "code": -1003, 
              "msg": "Way too much request weight used; IP banned until 1744874068494. Please use WebSocket Streams for live updates to avoid bans."
          }
          ```
        - El número que aparece después de until es el *timestamp* (milisegundos) de cuándo se levantará el castigo, permitiéndonos pausar el algoritmo de descarga hasta cumplirlo.
          
    - **Errores 5XX:** Son errores por parte del servidor. Por defecto, esperamos un minuto y reintentamos.

3. Otros límites:

- **RAW_REQUESTS:** Límite de llamadas brutas (en la corrida documentada, `exchangeInfo` devuelve **300,000** cada 5 minutos). Dado que el límite de *weight* (6,000/min) es mucho más estricto, es matemáticamente imposible alcanzar este techo respetando el peso.

- **ORDERS:** Límite de órdenes de trading (compras / ventas). Al ser este un script de solo lectura de datos históricos, este límite no aplica y será ignorado por ahora en la lógica de control.

In [3]:
EXCHANGE_URL = "https://api.binance.com/api/v3/exchangeInfo"

# Mapea unidad de intervalo a letra del header
INTERVAL_TO_LETTER = {"SECOND": "S", "MINUTE": "M", "HOUR": "H", "DAY": "D"}

# Traduce unidad de intervalo a segundos
INTERVAL_TO_SECONDS = {"SECOND": 1, "MINUTE": 60, "HOUR": 3600, "DAY": 86400}

print(f"Consultando los límites dinámicos de la API de Binance en {EXCHANGE_URL}")

API_LIMITS = {}

try:
    response = session.get(EXCHANGE_URL)
    response.raise_for_status()
    data = response.json()

    for lim_dic in data["rateLimits"]:
        limit_type = lim_dic["rateLimitType"]
        interval_unit = lim_dic["interval"]
        interval_num = lim_dic["intervalNum"]
        limit_value = lim_dic["limit"]

        limit_data = {
            "interval": interval_unit,
            "intervalNum": interval_num,
            "limit": limit_value,
        }

        if limit_type == "REQUEST_WEIGHT":
            letter = INTERVAL_TO_LETTER.get(interval_unit, interval_unit[0])
            # Arma el nombre del header de peso usado
            header_name = f"X-MBX-USED-WEIGHT-{interval_num}{letter}"

            limit_data["header_name"] = header_name

            API_LIMITS[limit_type] = limit_data

        # Agrupa límites ORDERS (segundo y día)
        elif limit_type == "ORDERS":
            if limit_type not in API_LIMITS:
                API_LIMITS[limit_type] = []
            API_LIMITS[limit_type].append(limit_data)

        else:
            API_LIMITS[limit_type] = limit_data

    print("\nConfiguración de límites obtenida correctamente.")
    print(API_LIMITS)

except Exception as e:
    print(f"Error al obtener los límites: {e}")
    # Aplica límites por defecto si falla exchangeInfo
    API_LIMITS["REQUEST_WEIGHT"] = {
        "limit": 6000,
        "header_name": "X-MBX-USED-WEIGHT-1M",
    }
    print("\nSe utilizarán límites por defecto (6000/1M) por seguridad.")

Consultando los límites dinámicos de la API de Binance en https://api.binance.com/api/v3/exchangeInfo

Configuración de límites obtenida correctamente.
{'REQUEST_WEIGHT': {'interval': 'MINUTE', 'intervalNum': 1, 'limit': 6000, 'header_name': 'X-MBX-USED-WEIGHT-1M'}, 'ORDERS': [{'interval': 'SECOND', 'intervalNum': 10, 'limit': 100}, {'interval': 'DAY', 'intervalNum': 1, 'limit': 200000}], 'RAW_REQUESTS': {'interval': 'MINUTE', 'intervalNum': 5, 'limit': 300000}}


### Función Auxiliar: cuenta regresiva visual


Dado que los tiempos de espera por límites de API (especialmente el error 418) pueden extenderse varios minutos, el uso simple de `time.sleep()` dejaría la consola congelada sin feedback.

Para mejorar la experiencia de usuario (UX), implementamos esta función auxiliar que muestra un cronómetro dinámico en la misma línea de la consola. Esto nos confirma que el script sigue activo y nos informa del tiempo restante exacto hasta el próximo intento.

In [4]:
def countdown(t: int):
    """
    Muestra una cuenta regresiva que se actualiza en la misma línea.

    Args:
        t (int): Segundos a esperar.
    """
    while t > 0:
        mins, secs = divmod(t, 60)
        timer = f"{mins:02d}:{secs:02d}"

        # Actualiza la misma línea con end='\r' y flush
        print(f"Pausa de seguridad {timer} restantes", end="\r", flush=True)

        time.sleep(1)
        t -= 1

    print("\nTiempo de espera concluido. Listo para reintentar.")

### Descarga de datos históricos


Definimos la función `download_safe_binance`. Esta función encapsula toda la lógica de paginación, control de errores y gestión preventiva de límites que hemos diseñado.

**Características de la implementación:**
* Paginación controlada: Iteramos bucle a bucle solicitando bloques de 1,000 velas (el máximo permitido), actualizando el puntero de tiempo (`startTime`) basándonos en el cierre de la última vela recibida.
* Prevención: A diferencia de los otros métodos, aquí inspeccionamos la cabecera `X-MBX-USED-WEIGHT` después de cada petición válida. Si superamos el 90% del límite asignado, forzamos una pausa preventiva para enfriar el contador antes de que Binance nos lance un error 429.
* La función integra además toda la lógica de seguridad (`countdown`, gestión de errores 429 / 418 / 5XX y de servidor) desarrollada en las secciones anteriores. Reintentando tras cada uno de estos errores o en caso de ser un error ajeno a estos el script se detiene inmediatamente para lanzar el error.

**Estructura de los datos:** Según la [documentación oficial de la API Binance](https://developers.binance.com/docs/binance-spot-api-docs/rest-api/market-data-endpoints), cada vela es devuelta como una lista de valores, donde el orden es posicional. Es vital conocer esta estructura para poder convertir a posteriori en un DataFrame.
```json
[
  [
  
    1499040000000,      // Kline open time (Inicio de la vela)
    "0.01634790",       // Open price (Precio de apertura)
    "0.80000000",       // High price (Precio máximo)
    "0.01575800",       // Low price (Precio mínimo)
    "0.01577100",       // Close price (Precio de cierre)
    "148976.11427815",  // Volume (Volumen)
    1499644799999,      // Kline Close time (Cierre de vela)
    "2434.19055334",    // Quote asset volume (Volumen del activo cotizado)
    308,                // Number of trades (Número de transacciones)
    "1756.87402397",    // Taker buy base asset volume (Volumen de compra Taker del activo base)
    "28.46694368",      // Taker buy quote asset volume (Volumen de compra Taker del activo cotizado)
    "0"                 // Unused field, ignore.
  
  ]
]
```


> **¿Qué es un "Taker"?**
>
> En los mercados financieros existen dos tipos de participantes:
> * *Makers* (creadores): Colocan órdenes limitadas que se quedan en el libro de órdenes esperando a ser ejecutadas. Aportan liquidez.
> * *Takers* (tomadores): Ejecutan órdenes a mercado (*market orders*) que se llenan instantáneamente contra una orden existente. Toman liquidez.
>
> **¿Por qué es importante el dato "Taker Buy Volume"?**
> A diferencia del volumen total, el *Taker Buy Volume* representa las compras agresivas. Si este valor es muy alto, indica que los compradores están impacientes y dispuestos a pagar el precio actual, lo cual es una señal alcista (*bullish*) de fuerza en el mercado.


In [5]:
def download_safe_binance(
    symbol: str, interval: str, start_time: int, limit: int
) -> list:
    """
    Descarga el historial completo de velas (klines) gestionando la paginación manual y respetando los límites de la API.

    Args:
        symbol (str): Par de trading (ej. 'BTCUSDT').
        interval (Str): Temporalidad (ej. '4h').
        start_time (int): Timestamp de inicio en milisegundos.
        limit (int): Velas por petición (el máximo es 1000).

    Returns:
        all_klines (list): Lista de listas con todos los datos históricos.
    """
    print(
        f"\nComenzando la descarga de datos de {symbol} en el intervalo de {interval}."
    )

    all_klines = []

    now = time.time() * 1000

    # Resuelve header y tope de peso desde API_LIMITS
    weight_header = API_LIMITS.get("REQUEST_WEIGHT", {}).get(
        "header_name", "X-MBX-USED-WEIGHT-1M"
    )
    weight_limit = API_LIMITS.get("REQUEST_WEIGHT", {}).get("limit", 6000)

    while start_time < now:
        try:
            params = {
                "symbol": symbol,
                "interval": interval,
                "startTime": start_time,
                "limit": limit,
            }

            response = session.get(BASE_URL, params=params)

            # Reintenta tras HTTP 429
            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", 60))
                print(f"\nError 429 límite excedido. Esperando {retry_after} segundos.")
                countdown(retry_after)
                continue

            # Reintenta tras HTTP 418 (ban de IP)
            elif response.status_code == 418:
                error_msg = response.json().get("msg", "")
                # Extrae timestamp de fin de ban del mensaje
                match = re.search(r"until (\d+)", error_msg)

                if match:
                    ban_lift = int(match.group(1))
                    wait_seconds = (ban_lift - int(time.time() * 1000)) / 1000
                    ban_lift_date = datetime.fromtimestamp(ban_lift / 1000)
                    print(
                        f"\nError crítico: IP baneada (error 418). La IP está bloqueada hasta: {ban_lift_date}"
                    )
                    countdown(wait_seconds + 10)

                else:
                    print(
                        f"\nError 418 con mensaje desconocido: {error_msg}. Esperando 5 minutos."
                    )
                    countdown(300)

                continue

            # Reintenta tras HTTP 5xx
            elif response.status_code >= 500:
                print(
                    f"\nError del servidor: {response.status_code}. Esperando 1 minuto."
                )
                countdown(60)
                continue

            response.raise_for_status()

            klines = response.json()

            # Termina si la página viene vacía
            if not klines:
                print("\nTodos los datos disponibles han sido descargados.")
                break

            all_klines.extend(klines)

            # Avanza start_time al cierre de la última vela + 1 ms
            start_time = klines[-1][6] + 1

            last_date = datetime.fromtimestamp(klines[-1][6] / 1000, tz=timezone.utc)
            print(
                f"Descargadas {len(all_klines)} velas. Última fecha obtenida: {last_date.strftime('%Y-%m-%d %H:%M')}.",
                end="\r",
                flush=True,
            )

            # Pausa si el peso supera el 90 % del límite
            if weight_header in response.headers:
                used_weight = int(response.headers[weight_header])

                if used_weight > (0.9 * weight_limit):
                    interval_unit = API_LIMITS.get("REQUEST_WEIGHT", {}).get(
                        "interval", "MINUTE"
                    )
                    interval_num = API_LIMITS.get("REQUEST_WEIGHT", {}).get(
                        "intervalNum", 1
                    )
                    base_wait = interval_num * INTERVAL_TO_SECONDS.get(
                        interval_unit, 60
                    )
                    pct_weight_used = used_weight / weight_limit * 100
                    print(
                        f"\nAlcanzado alto consumo del límite ({pct_weight_used:.2f}%). Esperando {base_wait} segundos."
                    )
                    countdown(base_wait)

        except requests.exceptions.RequestException as e:
            print(
                f"Error al conectar con la API de Binance: {e}. Reiniciando el proceso en 5 minutos."
            )
            countdown(360)
            continue

    print("\nTodos los datos disponibles han sido descargados.")

    return all_klines

### Procesamiento y limpieza de datos

Una vez descargados los datos crudos (*raw data*) en formato de lista, los vamos a procesar para realizar posteriormente el análisis cuantitativo. La función creada `raw_data_to_df` realiza las siguientes tareas:

1. **Estructuración:** Convierte la lista de listas en un `pandas.DataFrame` con los nombres de columnas apropiados.
2. **Selección:** Filtramos las columnas que consideramos relevantes.
3. **Conversión de tipos (*type casting*):**
    - La API de Binance nos devuelve todos los datos en formato `string`.
    - Convertimos `NumTrades` a entero (`int`).
    - Convertimos precios y volúmenes a punto flotante (`float`).
4. **Indexación temporal:** Transforma el *unix timestamp* (milisegundos desde 1970) a un formato de fecha legible `OpenTimestamp` y lo establece como el índice del DataFrame.
5. **Persistencia:** Guardamos una copia en formato `.csv` para su almacenamiento seguro.

In [6]:
def raw_data_to_df(
    klines_list: list, symbol: str, interval: str, origin: str
) -> pd.DataFrame:
    """
    Transforma la lista cruda de las velas de Binance en un DataFrame limpio, convierte tipos de datos,
    lo devuelve y crea un archivo .csv.

    Args:
        klines_list (list): Lista bruta de la API.
        symbol (str): Par de trading en formato BTCUSDT.
        interval (str): Temporalidad.
        origin (str): Identificador del método utilizado.

    Returns:
        df (pd.DataFrame): DataFrame procesado con índice temporal.
    """

    # Asigna nombres de columna del esquema klines
    col_names = [
        "OpenTime",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "CloseTime",
        "QuoteAssetVolume",
        "NumTrades",
        "TakerBuyBaseAssetVolume",
        "TakerBuyQuoteAssetVolume",
        "Ignore",
    ]

    cols_to_select = [
        "OpenTime",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "NumTrades",
        "TakerBuyBaseAssetVolume",
    ]

    df = pd.DataFrame(klines_list, columns=col_names)

    df = df[cols_to_select].copy()

    # Parsea OpenTime a datetime UTC
    df["OpenTimestamp"] = pd.to_datetime(
        pd.to_numeric(df["OpenTime"]), unit="ms", utc=True
    )

    # Establece OpenTimestamp como índice
    df.set_index("OpenTimestamp", inplace=True)

    df.drop("OpenTime", axis=1, inplace=True)

    df["NumTrades"] = df["NumTrades"].astype(int)

    # Castea precios y volúmenes a float
    float_cols = ["Open", "High", "Low", "Close", "Volume", "TakerBuyBaseAssetVolume"]
    df[float_cols] = df[float_cols].astype(float)

    # Exporta CSV con nombre derivado del símbolo
    file_path = f"{symbol}_{interval}_{origin}.csv"
    df.to_csv(file_path)
    print(f"\nArchivo {file_path} creado.")

    return df

### Ejecución del flujo de trabajo

Finalmente, unificamos toda la lógica desarrollada para ejecutar la descarga masiva de datos. En este bloque realizamos la iteración para los pares definidos. En cada uno de estos, descargamos todos sus datos disponibles, mostramos visualmente las dos primeras y últimas filas; y finalmente medimos el tiempo de ejecución para poder valorar su rendimiento.

In [7]:
print(
    f"Iniciando proceso de descarga de los {len(SYMBOLS)} pares definidos vía API directa."
)
start_time = time.time()

for symbol in SYMBOLS:
    klines = download_safe_binance(symbol, INTERVAL, START_TIME, LIMIT)

    df = raw_data_to_df(klines, symbol, INTERVAL, "BinanceAPI")

    print(f"\nDatos de {symbol} listos. Muestra:")
    display(df.iloc[[0, 1, -2, -1]])
    print("-" * 50)

# Resume la duración total
end_time = time.time()
total_time = end_time - start_time
mins, secs = divmod(total_time, 60)

print(f"\nProceso finalizado con éxito.")
_min_lbl = "minuto" if int(mins) == 1 else "minutos"
_sec_lbl = "segundo" if int(secs) == 1 else "segundos"
print(f"Tiempo total de ejecución: {int(mins)} {_min_lbl} y {int(secs)} {_sec_lbl}.")

Iniciando proceso de descarga de los 5 pares definidos vía API directa.

Comenzando la descarga de datos de BTCUSDT en el intervalo de 4h.
Descargadas 19261 velas. Última fecha obtenida: 2026-06-03 23:59.
Todos los datos disponibles han sido descargados.

Archivo BTCUSDT_4h_BinanceAPI.csv creado.

Datos de BTCUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2017-08-17 04:00:00+00:00,4261.48,4349.99,4261.32,4349.99,82.088865,334,64.013727
2017-08-17 08:00:00+00:00,4333.32,4485.39,4333.32,4427.30,63.619882,248,58.787633
2026-06-03 16:00:00+00:00,66076.01,66373.18,65251.00,65462.00,4242.738310,1109866,1923.941500
2026-06-03 20:00:00+00:00,65461.99,65860.00,64276.62,64311.35,5519.793080,1285775,2278.803540


--------------------------------------------------

Comenzando la descarga de datos de ETHUSDT en el intervalo de 4h.
Descargadas 19261 velas. Última fecha obtenida: 2026-06-03 23:59.
Todos los datos disponibles han sido descargados.

Archivo ETHUSDT_4h_BinanceAPI.csv creado.

Datos de ETHUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2017-08-17 04:00:00+00:00,301.13,307.96,298.00,307.96,1561.95305,711,1260.38649
2017-08-17 08:00:00+00:00,307.95,312.00,307.00,308.95,1177.71088,775,1093.84885
2026-06-03 16:00:00+00:00,1837.22,1844.08,1795.23,1803.41,104394.73990,1192028,45819.58280
2026-06-03 20:00:00+00:00,1803.41,1852.20,1769.62,1817.17,131099.11480,1620093,65295.93250


--------------------------------------------------

Comenzando la descarga de datos de SOLUSDT en el intervalo de 4h.
Descargadas 12737 velas. Última fecha obtenida: 2026-06-03 23:59.
Todos los datos disponibles han sido descargados.

Archivo SOLUSDT_4h_BinanceAPI.csv creado.

Datos de SOLUSDT listos. Muestra:


...

--------------------------------------------------

Comenzando la descarga de datos de XRPUSDT en el intervalo de 4h.
Descargadas 17707 velas. Última fecha obtenida: 2026-06-03 23:59.
Todos los datos disponibles han sido descargados.

Archivo XRPUSDT_4h_BinanceAPI.csv creado.

Datos de XRPUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2018-05-04 08:00:00+00:00,0.50000,1.5000,0.50000,0.91596,11308597.83,20069,4103379.71
2018-05-04 12:00:00+00:00,0.91596,0.9245,0.87644,0.88389,5279461.96,9892,2526556.28
2026-06-03 16:00:00+00:00,1.22520,1.2287,1.20200,1.20430,19216754.80,172226,9384135.30
2026-06-03 20:00:00+00:00,1.20420,1.2205,1.18680,1.20310,19876700.30,192003,9926633.00


--------------------------------------------------

Comenzando la descarga de datos de BNBUSDT en el intervalo de 4h.
Descargadas 18776 velas. Última fecha obtenida: 2026-06-03 23:59.
Todos los datos disponibles han sido descargados.

Archivo BNBUSDT_4h_BinanceAPI.csv creado.

Datos de BNBUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2017-11-06 00:00:00+00:00,1.50,1.799,0.50,1.700,649.120,33,207.450
2017-11-06 04:00:00+00:00,1.30,1.681,1.30,1.625,52482.550,357,22743.920
2026-06-03 16:00:00+00:00,633.00,633.600,620.00,622.770,46595.810,224007,20735.542
2026-06-03 20:00:00+00:00,622.77,630.000,616.21,621.630,54749.987,191983,21276.340


--------------------------------------------------

Proceso finalizado con éxito.
Tiempo total de ejecución: 0 minutos y 28 segundos.


### Análisis

Tras desarrollar el sistema completo de extracción (ETL) interactuando directamente con los endpoints de Binance, vamos a evaluar las fortalezas y debilidades de este enfoque:

**Ventajas:**
- Control absoluto y transparencia: Al no usar intermediarios, sabemos exactamente en todo momento qué petición se envía, qué cabeceras se reciben y cómo se gestionan los tiempos. Siendo esto vital para depurar posibles errores.
- Eficiencia y ligereza: Nuestro script solo requiere instalar `requests` y `pandas`, el resto de librerías son nativas. Esto hace que el script sea rápido de iniciar y fácil de desplegar en cualquier servidor.
- Resiliencia personalizada: Hemos implementado una lógica de protección diseñada específicamente para nuestras necesidades.
- Actualización inmediata: Si Binance lanza una nueva funcionalidad o cambia un endpoint, podemos adaptarnos editando el código.

**Desventajas:**
- Mantenimiento: Hemos tenido que escribir más de 50 líneas de código para gestionar el proceso de extracción de los datos.
- Complejidad: El manejo del bucle `while` y los `timestamps` es propenso a errores humanos.
- Fragilidad ante cambios en la API: Si Binance decide cambiar el nombre del header, nuestro script dejará de funcionar hasta que lo detectemos y lo corrijamos manualmente.

## Librería python-binance

Tras analizar la implementación manual, exploraremos ahora el uso de un *wrapper*. Estas librerías actúan como una capa de abstracción que gestiona automáticamente la construcción de URLs, la firma criptográfica y en muchos casos el manejo de errores básicos.

Utilizaremos `python-binance`, una librería de código abierto creada y mantenida por Sam McHardy.

### Documentación y GitHub

- **Documentación oficial:** https://python-binance.readthedocs.io/en/latest/
- **Repositorio de GitHub:** https://github.com/sammchardy/python-binance

Al ser una librería externa, requiere de instalación.

In [8]:
# Instala python-binance si falta la dependencia
!pip install python-binance


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### Importación y parámetros:

**Importación:** 

En este enfoque, sustituimos la gestión manual de peticiones HTTP por las clases específicas de la librería:
- `binance.client.Client`: Es la clase principal que actúa como puente con la API. A través de una instancia de este cliente, tendremos acceso a los métodos de descarga.
- `binance.exceptions.BinanceAPIException`: Nos permite capturar y gestionar errores específicos de Binance de forma limpia, sin necesidad de parsear códigos de estado HTTP manualmente.
- Reutilizamos las siguientes librerías: `pandas`, `re`, `time`, `datetime` y `IPython.display`.
  
**Parámetros y constantes:** 

Mantenemos `SYMBOLS` de los parámetros previamente declarados.

- `INTERVAL`: Podemos hacer uso de las constantes predefinidas. En este caso usaremos `Client.KLINE_INTERVAL_4HOUR`.
    - [Constantes disponibles](https://python-binance.readthedocs.io/en/latest/constants.html)
- `START_TIME`: Otra de las ventajas es que podemos declarar el parámetro de fecha de inicio con lenguaje natural.

In [9]:
from binance.client import Client
from binance.exceptions import BinanceAPIException
from requests.exceptions import RequestException

# Crea cliente público sin API keys
client = Client("", "")

INTERVAL = client.KLINE_INTERVAL_4HOUR

START_TIME = "1 Jan, 2017"

print("Configuración completada. Cliente python-binance inicializado.")

Configuración completada. Cliente python-binance inicializado.


### Comprobación de conectividad y latencia

- **Método:** `client.ping()`
- **Retorno esperado:** Si la conexión es exitosa devuelve un diccionario vacío `{}`.
- **Latencia:** Al no tener un atributo para medir la latencia lo hacemos manualmente con `time.time()`.
- **Documentación:** https://python-binance.readthedocs.io/en/latest/general.html#id1

In [10]:
print("Iniciando prueba de conexión con python-binance.")

try:
    start_time = time.time()

    client.ping()

    end_time = time.time()
    latency_ms = (end_time - start_time) * 1000

    print("Conexión exitosa.")
    print(f"\t- Latencia: {latency_ms:.2f} ms.")

except BinanceAPIException as e:
    print(f"Error de conexión con la API: {e}")

Iniciando prueba de conexión con python-binance.
Conexión exitosa.
	- Latencia: 1708.78 ms.


### Gestión de los límites: cambio de paradigma (reactivo vs. proactivo)

En este bloque nos encontramos con una diferencia fundamental en la arquitectura respecto al método manual. Al utilizar la función de alto nivel `client.get_historical_klines()`, la librería inicia su propio bucle interno para descargar todos los datos solicitados en una sola llamada de función.

Como consecuencia perdemos la granularidad. No podemos detener el proceso entre páginas para vigilar la cabecera `X-MBX-USED-WEIGHT`, por lo que la creación de un diccionario de límites y contadores manuales carece de sentido.

En su lugar, adoptaremos una estrategia reactiva. Lanzamos la petición completa y gestionamos los errores solo si ocurren mediante bloques `try-except`. Para ello, es vital conocer las excepciones específicas que define la librería:

1.  **`BinanceAPIException`**: Es la clase principal para errores de la API (Respuestas HTTP 4XX y 5XX).
    * Nos permite acceder a atributos clave como `e.status_code` (para detectar el 429/418) y `e.message` para leer el tiempo de baneo.
2.  **`requests.exceptions.RequestException`**: Para capturar fallos de red donde no se recibe respuesta del servidor.

* **Documentación de excepciones:** https://python-binance.readthedocs.io/en/latest/exceptions.html

### Descarga de datos históricos

Definimos la función `download_safe_python_binance`. A diferencia de la versión manual (*low level*) donde construíamos las URLs, aquí delegamos la paginación a la librería y nos centraremos exclusivamente en la resilencia.

**Características de esta implementación:**

1. **Enfoque todo o nada:** Utilizamos el método ya comentado previamente `client.get_historical_klines`. Esta función es síncrona y bloqueante, intenta descargar todo el rango de fechas solicitado de una sola vez.
3. **Persistencia condicional:** Capturamos para reintentar de forma selectiva los errores, igual que en la implementación previa.

In [11]:
def download_safe_python_binance(symbol: str, interval: str, start_time: str) -> list:
    """
    Descarga el historial completo de velas (klines) usando la librería python-binance.

    Args:
        symbol (str): Par de trading (ej. 'BTCUSDT').
        interval (str): Temporalidad (ej. '4h').
        start_time (str): Fecha en la que comenzar a descargar los datos.

    Returns:
        klines (list): Lista de listas con todos los datos históricos.
    """
    print(
        f"\nComenzando la descarga de datos de {symbol} en el intervalo de {interval}."
    )

    while True:
        try:
            klines = client.get_historical_klines(symbol, interval, start_time)
            print("\nTodos los datos disponibles han sido descargados.")
            return klines

        except BinanceAPIException as e:
            # Reintenta tras HTTP 429
            if e.status_code == 429:
                retry_after = 60
                # Lee Retry-After de la respuesta si existe
                if e.response is not None and "Retry-After" in e.response.headers:
                    retry_after = int(e.response.headers.get("Retry-After", 60))

                print(f"\nError 429 límite excedido. Esperando {retry_after} segundos.")
                countdown(retry_after)
                continue

            # Reintenta tras HTTP 418
            elif e.status_code == 418:
                # Extrae timestamp de fin de ban del mensaje
                match = re.search(r"until (\d+)", e.message)

                if match:
                    ban_lift = int(match.group(1))
                    wait_seconds = (ban_lift - (time.time() * 1000)) / 1000
                    ban_lift_date = datetime.fromtimestamp(ban_lift / 1000)
                    print(
                        f"\nError crítico IP baneada (error 418). La IP esta bloqueada hasta: {ban_lift_date}"
                    )
                    countdown(int(wait_seconds + 10))

                else:
                    print(
                        f"\nError 418 con mensaje desconocido: {e.message}. Esperando 5 minutos."
                    )
                    countdown(300)

                continue

            # Reintenta tras HTTP 5xx
            elif e.status_code >= 500:
                print(f"\nError del servidor: {e.message}. Esperando 1 minuto.")
                countdown(60)
                continue

            # Propaga errores de API no contemplados
            else:
                print(f"Error desconocido ({e.code}): {e.message}")
                raise e

        except RequestException as e:
            # Reintenta tras fallo de red
            print(f"Error de conexión: {e}. Reintentando en 1 minuto.")
            countdown(60)
            continue

### Procesamiento y limpieza de datos

Reutilizamos la función creada previamente para procesar y limpiar los datos ya que el input es el mismo.

### Ejecución del flujo de trabajo

Finalizamos ejecutando todo el flujo de trabajo de forma análoga a la previamente realizada.

In [12]:
print(
    f"Iniciando proceso de descarga de los {len(SYMBOLS)} pares definidos vía python-binance."
)
start_time = time.time()

for symbol in SYMBOLS:
    klines = download_safe_python_binance(symbol, INTERVAL, START_TIME)

    df = raw_data_to_df(klines, symbol, INTERVAL, "python_binance")

    print(f"\nDatos de {symbol} listos. Muestra:")
    display(df.iloc[[0, 1, -2, -1]])
    print("-" * 50)

# Resume la duración total
end_time = time.time()
total_time = end_time - start_time
mins, secs = divmod(total_time, 60)

print(f"\nProceso finalizado con éxito.")
_min_lbl = "minuto" if int(mins) == 1 else "minutos"
_sec_lbl = "segundo" if int(secs) == 1 else "segundos"
print(f"Tiempo total de ejecución: {int(mins)} {_min_lbl} y {int(secs)} {_sec_lbl}.")

Iniciando proceso de descarga de los 5 pares definidos vía python-binance.

Comenzando la descarga de datos de BTCUSDT en el intervalo de 4h.

Todos los datos disponibles han sido descargados.

Archivo BTCUSDT_4h_python_binance.csv creado.

Datos de BTCUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2017-08-17 04:00:00+00:00,4261.48,4349.99,4261.32,4349.99,82.088865,334,64.013727
2017-08-17 08:00:00+00:00,4333.32,4485.39,4333.32,4427.30,63.619882,248,58.787633
2026-06-03 16:00:00+00:00,66076.01,66373.18,65251.00,65462.00,4242.738310,1109866,1923.941500
2026-06-03 20:00:00+00:00,65461.99,65860.00,64276.62,64318.72,5537.186020,1289027,2287.821860


--------------------------------------------------

Comenzando la descarga de datos de ETHUSDT en el intervalo de 4h.

Todos los datos disponibles han sido descargados.

Archivo ETHUSDT_4h_python_binance.csv creado.

Datos de ETHUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2017-08-17 04:00:00+00:00,301.13,307.96,298.00,307.96,1561.95305,711,1260.38649
2017-08-17 08:00:00+00:00,307.95,312.00,307.00,308.95,1177.71088,775,1093.84885
2026-06-03 16:00:00+00:00,1837.22,1844.08,1795.23,1803.41,104394.73990,1192028,45819.58280
2026-06-03 20:00:00+00:00,1803.41,1852.20,1769.62,1816.88,131291.63910,1623472,65399.74760


--------------------------------------------------

Comenzando la descarga de datos de SOLUSDT en el intervalo de 4h.

Todos los datos disponibles han sido descargados.

Archivo SOLUSDT_4h_python_binance.csv creado.

Datos de SOLUSDT listos. Muestra:


...

--------------------------------------------------

Comenzando la descarga de datos de XRPUSDT en el intervalo de 4h.

Todos los datos disponibles han sido descargados.

Archivo XRPUSDT_4h_python_binance.csv creado.

Datos de XRPUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2018-05-04 08:00:00+00:00,0.50000,1.5000,0.50000,0.91596,11308597.83,20069,4103379.71
2018-05-04 12:00:00+00:00,0.91596,0.9245,0.87644,0.88389,5279461.96,9892,2526556.28
2026-06-03 16:00:00+00:00,1.22520,1.2287,1.20200,1.20430,19216754.80,172226,9384135.30
2026-06-03 20:00:00+00:00,1.20420,1.2205,1.18680,1.20290,19886636.60,192491,9934057.20


--------------------------------------------------

Comenzando la descarga de datos de BNBUSDT en el intervalo de 4h.

Todos los datos disponibles han sido descargados.

Archivo BNBUSDT_4h_python_binance.csv creado.

Datos de BNBUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2017-11-06 00:00:00+00:00,1.50,1.799,0.50,1.700,649.120,33,207.450
2017-11-06 04:00:00+00:00,1.30,1.681,1.30,1.625,52482.550,357,22743.920
2026-06-03 16:00:00+00:00,633.00,633.600,620.00,622.770,46595.810,224007,20735.542
2026-06-03 20:00:00+00:00,622.77,630.000,616.21,621.810,54951.702,192977,21397.438


--------------------------------------------------

Proceso finalizado con éxito.
Tiempo total de ejecución: 0 minutos y 54 segundos.


### Análisis

En este bloque hemos sustituido la construcción manual de peticiones HTTP por el uso de un *wrapper*. A continuación, evaluamos las características intrínsecas de esta herramienta tras haberla implementado.

**Ventajas:**

- Abstracción: La mayor fortaleza es la capacidad para simplificar flujos complejos.
- Usabilidad: Las excepciones como objetos tipados, el uso de constantes y la implementación de fechas en lenguaje natural simplifican el manejo.

**Desventajas:**

- Arquitectura de caja negra: Al delegar la lógica de descarga a la librería, perdemos visibilidad sobre el proceso interno. Forzándonos a cambiar nuestra estrategia mixta (proactiva y reactiva) a simplemente reactiva.
- Dependencia de terceros: Aunque es código abierto, introducimos una dependencia externa crítica. Si el proyecto es abandonado o los posibles cambios introducidos por Binance no son reflejados, el código podría quedar obsoleto o vulnerable.
- Rigidez en el manejo de errores: Estamos limitados a la estructura de error que define el *wrapper*. Si la API devuelve un error nuevo o no contemplado en el *wrapper*, podríamos perder información valiosa del error original.

## Librería ccxt 

Finalmente, exploraremos el estándar de la industria para el desarrollo de bots de trading multiplataforma, ccxt.

A diferencia de la solución anterior, ccxt no es un *wrapper* exclusivo de Binance. Es una librería masiva que normaliza la interacción con más de 100 casas de cambio (Binance, Kraken, Coinbase, OKX, etc.). Esto significa que, teóricamente, podríamos cambiar nuestro bot de Binance a Kraken cambiando unas pocas líneas de código.

### Documentación y GitHub

* **Documentación oficial:** https://docs.ccxt.com/
* **Repositorio de GitHub:** https://github.com/ccxt/ccxt

Como en el caso anterior requiere de instalación.

> **Nota de rendimiento:** Esta librería es significativamente más pesada (*overhead*) porque incluye los controladores de más de 100 exchanges. Aunque solo utilicemos Binance, la instalación y la carga en memoria inicial son mayores.

In [13]:
# Instala ccxt si falta la dependencia
!pip install ccxt


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### Importación y parámetros

**Importación:**

En este caso, también sustituimos la gestión manual de las peticiones por la clase de la librería.

- `ccxt`: Nos da acceso al constructor `ccxt.binance()`.
- Reutilizamos las siguientes librerías: `pandas`, `re`, `time`,  `datetime` y `IPython.display`.

**Parámetros:**

Al usar el método implícito `public_get_klines` (por razones que explicaremos a continuación), los parámetros deben respetar el formato nativo de Binance. Mantenemos así la definición original de los parámetros `SYMBOLS`, `INTERVAL` y  `LIMIT`.

- `START_TIME`: Aprovecharemos la utilidad [`parse8601`](https://docs.ccxt.com/?id=working-with-datetimes-and-timestamps) de ccxt, que convierte una fecha legible (ISO 8601) directamente al timestamp en milisegundos que exige la API, ahorrándonos el uso de la librería `datetime` para la conversión.

In [14]:
import ccxt

INTERVAL = "4h"

# Configura Binance con rate limiting de ccxt
binance = ccxt.binance({"enableRateLimit": True})

start_str = "2017-01-01T00:00:00Z"
START_TIME = binance.parse8601(start_str)

print("Configuración completada. Exchange inicializado con rate limiter.")

Configuración completada. Exchange inicializado con rate limiter.


### Comprobación de conectividad y latencia

En lugar de realizar un simple `ping` (como hicimos en los métodos anteriores), aprovecharemos la capacidad de abstracción de ccxt.

* **Método:** `exchange.fetch_status()`.
    * **Ventaja:** Este método no solo comprueba que tengamos conexión con el servidor, sino que consulta el estado interno del motor de emparejamiento (*matching engine*).
    * **Normalización:** ccxt traduce la respuesta específica de Binance a un formato estándar: `{'status': 'ok'}` o `{'status': 'maintenance'}`.
> **Nota de sintaxis:** ccxt es una librería multiplataforma. Mientras que los ejemplos generales suelen estar en `camelCase` (`fetchStatus`), para **Python** la documentación exige el uso de **`snake_case`**.
>
> Por esta razón, escribiremos **`exchange.fetch_status()`**.
> * **Documentación:** https://docs.ccxt.com/?id=api-method-naming-conventions
* **Retorno esperado:** Un diccionario con la clave `'status': 'ok'`.
* **Latencia:** Seguiremos midiéndola manualmente (tiempo de ida y vuelta de la petición).
* **Documentación:**
    * https://docs.ccxt.com/baseSpec?id=fetchstatus
    * https://docs.ccxt.com/?id=exchange-status

In [15]:
print("Iniciando prueba de conexión con ccxt.")

try:
    start_time = time.time()

    response = binance.fetch_status()

    end_time = time.time()
    latency_ms = (end_time - start_time) * 1000

    if response["status"] == "ok":
        print("Conexión exitosa.")
        print(f"\t- Latencia: {latency_ms:.2f} ms.")
    else:
        print(f"El sistema reporta el siguiente estado: {response['status']}")
        # Muestra ETA de mantenimiento si viene en status
        if response["status"] == "maintenance" and response["eta"]:
            print(f"Fin del mantenimiento: {response['eta']}")

except ccxt.NetworkError as e:
    print(f"Error de red: {e}")
except ccxt.ExchangeError as e:
    print(f"Error del exchange: {e}")

Iniciando prueba de conexión con ccxt.
Conexión exitosa.
	- Latencia: 282.48 ms.


### Gestión de los límites

En este enfoque con ccxt, nos encontramos con la solución más avanzada de las tres. Al haber instanciado el cliente activando la opción `'enableRateLimit': True`, activamos el rate limiter interno de la librería, delegando la gestión de los límites.

> **¿Cómo funciona?** ccxt implementa un algoritmo de *token bucket*. La librería conoce los pesos y límites de Binance. Antes de enviar la petición `public_get_klines`, calcula si tiene "fichas" disponibles. Si no tiene, el script se pausará automáticamente el tiempo necesario antes de lanzar la siguiente llamada.
>
> **Documentación:** https://docs.ccxt.com/?id=rate-limit

**Diferencias clave:**
1. No es necesario crear un diccionario `API_LIMITS` ni leer cabeceras `X-MBX-USED-WEIGHT`. Confiamos en que la librería gestione las llamadas por nosotros.
2. Aunque usamos un método para acceder al endpoint en crudo `public_get_klines` (frente a `fetch_ohlcv`), al hacerlo a través de la instancia `binance`, el rate limiter intercepta la llamada igualmente.

**Jerarquía de excepciones:** 
A pesar de la automatización, debemos protegernos contra fallos de red o baneos si el limitador interno falla. ccxt normaliza los errores de todos los exchanges con la siguiente jerarquía:
* `ccxt.NetworkError`: Equivalente a `RequestException`. Fallos de conexión, DNS o timeout. Reintentamos tras una pausa breve.
* `ccxt.ExchangeError`: Error genérico de la lógica del exchange (ej. símbolo mal formado). No lo vamos a capturar permitiremos al script detenerse para que corrijamos el código.
* `ccxt.RateLimitExceeded`: Es el error 429 (too many requests), indica que hemos superado el límite de peso momentáneo.
    * Accedemos a `e.response.headers['Retry-After']` para pausar el script exactamente los segundos que demanda Binance antes de continuar.

* `ccxt.DDoSProtection`: Error 418 (baneo de IP), ocurre si ignoramos repetidamente las advertencias. Es un bloqueo crítico que puede durar desde 2 minutos hasta 3 días.
    * En este caso, debemos analizar el mensaje de texto de la excepción (que contiene el JSON de Binance) para buscar la frase clave `banned until timestamp`. Extraeremos ese timestamp usando expresiones regulares (igual que en el método manual) para calcular el tiempo exacto de espera y detener el script hasta que el castigo expire.

* **Documentación:** https://docs.ccxt.com/?id=error-handling

### Descarga de datos históricos

Para la función de descarga `download_safe_ccxt`, nos enfrentamos a una decisión de diseño crítica. A continuación, justificamos por qué hemos utilizado el acceso al endpoint crudo:

* El problema: El método estándar `fetch_ohlcv` normaliza la respuesta eliminando columnas valiosas (número de trades, volumen de cotización, etc.).
* La solución: Utilizamos el método implícito `public_get_klines`. Con este método accedemos al endpoint crudo de Binance, devolviéndonos las 12 columnas originales.

**Características de la implementación:**
* Paginación manual: A diferencia de `python-binance`, aquí debemos calcular el siguiente `startTime` sumando 1 ms al cierre de la última vela recibida. Además, al usar el método crudo, ccxt no convierte los datos. Recibiremos *strings* que debemos convertir a enteros para controlar la paginación.
* Gestión de límites automática y manual: Confiamos en el rate limiter de ccxt para gestionar las llamadas, pero implementamos una red de seguridad (lectura de *headers* y Regex) para los casos de bloqueo (429 / 418).

In [16]:
def download_safe_ccxt(
    exchange, symbol: str, interval: str, start_time: int, limit: int
) -> list:
    """
    Descarga el historial completo de velas (klines) usando la librería ccxt.

    Args:
        exchange: Instancia configurada de ccxt.binance (con enableRateLimit=True).
        symbol (str): Par de trading (ej. 'BTCUSDT').
        interval (str): Temporalidad (ej. '4h').
        start_time (int): Fecha parseada previamente a ms.
        limit (int): Velas por petición (el máximo es 1000).

    Returns:
        all_klines (list): Lista de listas con todos los datos históricos.
    """
    print(f"\nComenzando la descarga de datos de {symbol} en el intervalo {interval}.")

    all_klines = []

    now = time.time() * 1000

    # Normaliza start_time a entero en ms
    current_start = int(start_time)

    while current_start < now:
        try:
            params = {
                "symbol": symbol,
                "interval": interval,
                "startTime": current_start,
                "limit": limit,
            }

            klines = exchange.public_get_klines(params)

            # Termina si la página viene vacía
            if not klines:
                print("\nTodos los datos disponibles han sido descargados.")
                break

            all_klines.extend(klines)

            # Avanza current_start al cierre de la última vela + 1 ms
            last_timestamp = int(klines[-1][6])
            current_start = last_timestamp + 1

            last_date = datetime.fromtimestamp(last_timestamp / 1000, tz=timezone.utc)
            print(
                f"Descargadas {len(all_klines)} velas. Última fecha obtenida: {last_date.strftime('%Y-%m-%d %H:%M')}.",
                end="\r",
                flush=True,
            )

        # Reintenta tras RateLimitExceeded
        except ccxt.RateLimitExceeded as e:
            retry_after = 60
            if (
                hasattr(e, "response")
                and e.response
                and "Retry-After" in e.response.headers
            ):
                retry_after = int(e.response.headers.get("Retry-After", 60))
            print(f"\nError 429 límite excedido. Esperando {retry_after} segundos.")
            countdown(retry_after)
            continue

        # Reintenta tras DDoSProtection (ban de IP)
        except ccxt.DDoSProtection as e:
            match = re.search(r"until (\d+)", str(e))
            if match:
                ban_lift = int(match.group(1))
                wait_seconds = (ban_lift - (time.time() * 1000)) / 1000
                ban_lift_date = datetime.fromtimestamp(ban_lift / 1000)
                print(
                    f"\nError crítico: IP baneada (error 418). La IP está bloqueada hasta: {ban_lift_date}"
                )
                countdown(wait_seconds + 10)

            else:
                print(f"\nError 418 con mensaje desconocido: {e}. Esperando 5 minutos.")
                countdown(300)

            continue

        except ccxt.NetworkError as e:
            print(f"\nError de red: {e}. Reintentando en 1 minuto.")
            countdown(60)

    return all_klines

### Procesamiento y limpieza de datos

Reutilizamos la función creada previamente para procesar y limpiar los datos ya que el input es el mismo.

### Ejecución del flujo de trabajo

Finalizamos ejecutando todo el flujo de trabajo de forma análoga a la previamente realizada.

In [17]:
print(f"Iniciando proceso de descarga de los {len(SYMBOLS)} pares definidos vía CCXT.")
start_time = time.time()

for symbol in SYMBOLS:
    klines = download_safe_ccxt(binance, symbol, INTERVAL, START_TIME, LIMIT)

    df = raw_data_to_df(klines, symbol, INTERVAL, "ccxt")

    print(f"\nDatos de {symbol} listos. Muestra:")
    display(df.iloc[[0, 1, -2, -1]])
    print("-" * 50)

# Resume la duración total
end_time = time.time()
total_time = end_time - start_time
mins, secs = divmod(total_time, 60)

print(f"\nProceso finalizado con éxito.")
_min_lbl = "minuto" if int(mins) == 1 else "minutos"
_sec_lbl = "segundo" if int(secs) == 1 else "segundos"
print(f"Tiempo total de ejecución: {int(mins)} {_min_lbl} y {int(secs)} {_sec_lbl}.")

Iniciando proceso de descarga de los 5 pares definidos vía CCXT.

Comenzando la descarga de datos de BTCUSDT en el intervalo 4h.
Descargadas 19261 velas. Última fecha obtenida: 2026-06-03 23:59.
Archivo BTCUSDT_4h_ccxt.csv creado.

Datos de BTCUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2017-08-17 04:00:00+00:00,4261.48,4349.99,4261.32,4349.99,82.088865,334,64.013727
2017-08-17 08:00:00+00:00,4333.32,4485.39,4333.32,4427.30,63.619882,248,58.787633
2026-06-03 16:00:00+00:00,66076.01,66373.18,65251.00,65462.00,4242.738310,1109866,1923.941500
2026-06-03 20:00:00+00:00,65461.99,65860.00,64252.51,64313.77,5572.297420,1294513,2311.634430


--------------------------------------------------

Comenzando la descarga de datos de ETHUSDT en el intervalo 4h.
Descargadas 19261 velas. Última fecha obtenida: 2026-06-03 23:59.
Archivo ETHUSDT_4h_ccxt.csv creado.

Datos de ETHUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2017-08-17 04:00:00+00:00,301.13,307.96,298.00,307.96,1561.95305,711,1260.38649
2017-08-17 08:00:00+00:00,307.95,312.00,307.00,308.95,1177.71088,775,1093.84885
2026-06-03 16:00:00+00:00,1837.22,1844.08,1795.23,1803.41,104394.73990,1192028,45819.58280
2026-06-03 20:00:00+00:00,1803.41,1852.20,1769.62,1818.28,131533.62270,1627017,65594.26620


--------------------------------------------------

Comenzando la descarga de datos de SOLUSDT en el intervalo 4h.
Descargadas 12737 velas. Última fecha obtenida: 2026-06-03 23:59.
Archivo SOLUSDT_4h_ccxt.csv creado.

Datos de SOLUSDT listos. Muestra:


...

--------------------------------------------------

Comenzando la descarga de datos de XRPUSDT en el intervalo 4h.
Descargadas 17707 velas. Última fecha obtenida: 2026-06-03 23:59.
Archivo XRPUSDT_4h_ccxt.csv creado.

Datos de XRPUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2018-05-04 08:00:00+00:00,0.50000,1.5000,0.50000,0.91596,11308597.83,20069,4103379.71
2018-05-04 12:00:00+00:00,0.91596,0.9245,0.87644,0.88389,5279461.96,9892,2526556.28
2026-06-03 16:00:00+00:00,1.22520,1.2287,1.20200,1.20430,19216754.80,172226,9384135.30
2026-06-03 20:00:00+00:00,1.20420,1.2205,1.18680,1.20360,19920589.10,192952,9963566.40


--------------------------------------------------

Comenzando la descarga de datos de BNBUSDT en el intervalo 4h.
Descargadas 18776 velas. Última fecha obtenida: 2026-06-03 23:59.
Archivo BNBUSDT_4h_ccxt.csv creado.

Datos de BNBUSDT listos. Muestra:


,Open,High,Low,Close,Volume,NumTrades,TakerBuyBaseAssetVolume
OpenTimestamp,,,,,,,
2017-11-06 00:00:00+00:00,1.50,1.799,0.50,1.700,649.120,33,207.450
2017-11-06 04:00:00+00:00,1.30,1.681,1.30,1.625,52482.550,357,22743.920
2026-06-03 16:00:00+00:00,633.00,633.600,620.00,622.770,46595.810,224007,20735.542
2026-06-03 20:00:00+00:00,622.77,630.000,616.21,621.970,54964.982,193277,21399.632


--------------------------------------------------

Proceso finalizado con éxito.
Tiempo total de ejecución: 0 minutos y 27 segundos.


### Análisis

Tras implementar la solución utilizando la librería ccxt en su modalidad mixta (gestión automática de límites + endpoint implícito), evaluamos su impacto en la arquitectura del proyecto:

**Ventajas:**
* Gestión de límites autopilot: Sin duda, la mayor fortaleza. Al activar `enableRateLimit: True`, delegamos la compleja gestión de los límites y los tiempos de espera al algoritmo *token bucket* interno de la librería. Esto simplifica enormemente el código, eliminando la necesidad de contadores manuales y reduciendo el riesgo de error humano en el cálculo de las pausas.
* Normalización de errores: ccxt traduce los diferentes códigos de error específicos de cada exchange (Binance, Kraken, Coinbase, etc.) a una jerarquía estándar (`NetworkError`, `RateLimitExceeded`, `ExchangeError`). De esta forma, nuestros bloques `try-except` funcionan independientemente de la plataforma.
* Flexibilidad híbrida: Nos permite usar la infraestructura de seguridad de la librería, pero pudiendo acceder a los métodos crudos para obtener la totalidad de los datos sin sufrir la pérdida de información que impone el formato estandarizado.
* Portabilidad: Aunque en este caso hemos usado métodos específicos de Binance, la estructura de conexión y configuración es estándar. Migrar este script a otro exchange requeriría cambios mínimos.

**Desventajas:**
* Inconsistencia de tipos: Al usar la API implícita perdemos la conversión automática de tipos. ccxt nos devuelve los datos tal cual los envía Binance (strings), siendo necesarias las conversiones manuales.
* Dependencia de actualizaciones: La gestión de los límites funciona porque ccxt tiene hardcodeadas las reglas de los pesos de Binance. Si Binance cambia sus pesos mañana y la librería no se actualiza, nuestro script podría empezar a fallar.
* Curva de aprendizaje y documentación: La librería es inmensa y su documentación, al ser multiplataforma (JS, Python, PHP), puede generar confusión con las convenciones de los nombres o la ubicación de métodos específicos.
* Sobrecarga: Cargar una librería que soporta más de 100 exchanges consume más memoria y tiempo de inicio que un script ligero con `requests`, aunque en nuestro caso de uso es despreciable.

## Análisis final

Tras haber implementado y analizado las tres estrategias distintas para la extracción de datos históricos, contrastamos latencia, tiempo de descarga y control operativo para decidir el cliente del ETL productivo.

**Métricas de la corrida documentada** (salidas embebidas del cuaderno; *timeframe* 4h, cinco pares, inicio 2017-01-01 UTC):

| Enfoque | Latencia de prueba (ping) | Tiempo total descarga histórica |
| :--- | :--- | :--- |
| `requests` (manual) | **1,973.21** ms | **1** minuto y **2** segundos |
| `python-binance` | **345.24** ms | **1** minuto y **38** segundos |
| `ccxt` | **980.85** ms | **1** minuto y **2** segundos |

En las tres rutas la última vela alcanzada en los prints de descarga es **2026-05-22 23:59** UTC (p. ej. **19,189** velas en `BTCUSDT` con `requests`).

| Criterio | **1. Manual (`requests`)** | **2. Específico (`python-binance`)** | **3. Universal (`ccxt`)** |
| :--- | :--- | :--- | :--- |
| **Nivel de abstracción** | Bajo (raw HTTP) | Medio | Alto (unificado) |
| **Control de datos** | Total (todo lo que envía la API) | Alto (objetos propios) | Variable (normalizado vs. crudo) |
| **Gestión de límites** | Reactiva y proactiva (algoritmo propio) | Reactiva (exceptions) | Parte automática (*token bucket*) y en caso de que falle reactiva (exceptions) |
| **Complejidad de código** | Alta | Media | Media/baja |
| **Dependencias** | Mínimas (`requests`) | Específica de Binance | Pesada (soporte +100 exchanges) |
| **Resiliencia** | Personalizada | Depende de la librería | Muy alta (estandarizada) |
| **Escalabilidad** | Nula (requiere reescribir) | Nula | Total (cambio de configuración) |

Para un entorno de producción comercial o un bot de trading que opere en múltiples mercados, la elección lógica sería ccxt. Su capacidad para estandarizar errores y gestionar los *rate limits* automáticamente ahorra cientos de horas de desarrollo y mantenimiento.

Sin embargo, para el contexto de este TFG, hemos decidido el método manual.

Justificación de la elección:
1. Independencia tecnológica: Al no depender de librerías de terceros, eliminamos el riesgo de obsolescencia externa. Si Binance actualiza su API, no necesitamos esperar a que los creadores de la librería la actualicen, podemos adaptar nuestro código inmediatamente.
2. Transparencia y depuración: En un contexto científico, es vital entender cómo se obtienen los datos. El método manual nos obliga a interactuar directamente con las cabeceras HTTP (`X-MBX-USED-WEIGHT`), dándonos una visibilidad total sobre el comportamiento del servidor y la integridad de los datos recibidos.
3. Valor pedagógico: Construir el motor de extracción desde cero (gestionando la paginación, tiempos y excepciones) nos ayuda a asentar nuestro dominio profundo en la infraestructura de los datos, un objetivo central de este estudio.

**Nota.** Latencias, tiempos de descarga, conteos de velas y límites leídos de `exchangeInfo` dependen de la red y del estado de la API al ejecutar el cuaderno; en otra corrida pueden variar.